# Noisy Network
### リファレンス
- [Ｐｙｔｈｏｎで学ぶ強化学習](https://www.amazon.co.jp/dp/B082HNNGQG/)
- [DQNの進化史 ①DeepMindのDQN](https://horomary.hatenablog.com/entry/2021/01/26/233351)
- [pytorchでnoisy networkを実装](https://jsapachehtml.hatenablog.com/entry/2018/10/13/173303)

### code
- [icoxfog417/baby-steps-of-rl-ja](https://github.com/icoxfog417/baby-steps-of-rl-ja)

### Factorized Gaussian Noise
ファクター化されたガウスノイズ（Factorized Gaussian Noise）は、ニューラルネットワークの重みやバイアスにノイズを組み込む手法であり、特に強化学習における探索（exploration）能力の向上に用いられます。この手法は、NoisyNet（Noisy Networks for Exploration）という論文で提案され、従来の探索手法に比べて効率的で効果的な探索を可能にします。

ファクター化の目的:  
重み行列全体に独立したノイズを適用すると、パラメータ数が膨大になり計算コストが高くなります。ファクター化されたガウスノイズでは、ノイズを入力と出力の次元に分解し、効率的にノイズを生成します。

ノイズの生成:  
入力ノイズ ϵin と出力ノイズ ϵout を以下のように生成します。
ϵin ~ N(0, 1)  
ϵout ~ N(0, 1)  

f(x) = sign(x)⋅ √x

重みのノイズ:
ϵW = f(ϵout) x f(ϵin) ^T

バイアスのノイズ:
ϵb = f(ϵout)

なぜ関数 f を使うのか:  
関数 f を適用することで、ノイズの分布がゼロ平均・一定分散を保ちつつ、ノイズ間の依存性を減らすことができます。これにより、重み行列全体のノイズを2つのベクトルの外積で表現でき、計算量を削減できます。

実装上のポイント:  
- 効率性: ファクター化により、ノイズの生成と適用が計算的に効率的になります。 
- 学習可能なパラメータ: μ と σ は学習によって最適化され、エージェントが環境に適応する能力を高めます。
- 探索の自動化: ノイズの強度が学習されるため、ハイパーパラメータの手動調整が不要になります。

​


In [5]:
import gymnasium as gym
import torch

from code.catcher_observer import CatcherObserver
from code.noisy_network_trainer import NoisyNetworkTrainer
from code.noisy_network_agent import NoisyNetworkAgent


In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('device: ', device)
Training = False

device:  mps


### Training

In [7]:
# BreakoutDeterministic-v4を使う理由:
# 必ず指示した通りの行動が実行され、高すぎるフレームレートを間引くため毎回4フレームスキップします。
# これ以外の環境だと指示した通りに動かなかったりフレームスキップ数がランダム化されたりと環境の遷移が確率的になり、
# 攻略難易度が劇的上昇します(v5であれば改善されているかもしれない)。
game = "BreakoutDeterministic-v4"

# 16349 steps / 84min
# A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
# [Powered by Stella]
# Done initialization. From now, begin training!
# episode: 1531, step_count: 20, sum_reward: 0.0, avg_loss: 0.0
# best model saved:  models/noisy_network_agent_best.pth
# episode: 1548, step_count: 75, sum_reward: 1.0, avg_loss: 0.0
# best model saved:  models/noisy_network_agent_best.pth
# episode: 1572, step_count: 125, sum_reward: 2.0, avg_loss: 0.0
# best model saved:  models/noisy_network_agent_best.pth
# episode: 1766, step_count: 142, sum_reward: 3.0, avg_loss: 0.0
# best model saved:  models/noisy_network_agent_best.pth
# episode: 2576, step_count: 26, sum_reward: 0.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_2576.pth
# episode: 2811, step_count: 27, sum_reward: 0.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_2811.pth
# episode: 2865, step_count: 55, sum_reward: 1.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_2865.pth
# episode: 4084, step_count: 210, sum_reward: 5.0, avg_loss: 0.0
# best model saved:  models/noisy_network_agent_best.pth
# episode: 4277, step_count: 28, sum_reward: 0.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_4277.pth
# episode: 6178, step_count: 30, sum_reward: 0.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_6178.pth
# episode: 9132, step_count: 57, sum_reward: 1.0, avg_loss: 0.0
# model saved:  models/noisy_network_agent_9132.pth
# episode: 9832, step_count: 202, sum_reward: 8.0, avg_loss: 0.01
# best model saved:  models/noisy_network_agent_best.pth
# ...
# episode: 16070, step_count: 266, sum_reward: 10.0, avg_loss: 0.01
# best model saved:  models/noisy_network_agent_best.pth
# episode: 16349, step_count: 27, sum_reward: 0.0, avg_loss: 0.02
# model saved:  models/noisy_network_agent_16349.pth

if Training:
    model_path = "models/noisy_network_agent.pth"
    trainer = NoisyNetworkTrainer(model_path=model_path, device=device)
    env = gym.make(game, render_mode="rgb_array")
    obs = CatcherObserver(env, 84, 84, 4) # width, height, n_frame
    agent = trainer.train(env=obs)

### Play

In [8]:
model_path = "models/noisy_network_agent_best.pth"
env = gym.make(game)
obs = CatcherObserver(env, 84, 84, 4) # width, height, n_frame
agent = NoisyNetworkAgent.load(actions=list(range(env.action_space.n)), model_path=model_path, device=device)
agent.play(obs, render=True)

0: Get reward 4.0.
1: Get reward 4.0.
2: Get reward 2.0.
3: Get reward 4.0.
4: Get reward 0.0.
5: Get reward 0.0.
6: Get reward 4.0.
7: Get reward 4.0.
8: Get reward 5.0.
9: Get reward 5.0.
